# Machine Learning - Predicting Success From Pre-Release Data

The target variable is 'success_combined', a weighted score of 60% sales, 15% metascore and 25% userscore, binarised at the median.

**Note:** an earlier version of this notebook included 'critic_score' as a feature, but since the success label partly depends on review scores, this created a circular relationship. The current version uses only pre-release features since we want to know whether the game will succeed before the release. To compensate for the loss of this strong predictor, feature engineering is applied to extract additional signal from the available pre-release data. Also the earlier version only had LogisticRegression and Decition Tree as models, this one includes Random Forest as well. Only the final version(this) of the notebook is pushed to github.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ( accuracy_score, classification_report, roc_auc_score, precision_score, recall_score, f1_score)
from sklearn.preprocessing import StandardScaler
from datetime import datetime

%matplotlib inline

In [3]:
df = pd.read_csv("C:/Users/George/Desktop/data/combined_label.csv")
print(f"Dataset shape: {df.shape}")

Dataset shape: (8624, 22)


## 1. Feature Engineering

The original four pre-release features(genre, console, publisher, release year) are weak predictors on their own. Several derived features are added to extract more signal from the same underlying data.

**Engineered Features:**
- 'publisher_game_count' - How many games each publisher has produced(measure of size and experience)
- 'publisher_success_rate' - Historical success rate by genre(training only)
- 'genre_success_rate' - Historical success rate by genre(training only)
- 'console_success_rate' - Historical success rate by platform(training only)
- 'game_age' - Years since release(captures how older vs newer titles perform)

**Important:** all success-rate featues are computed from the training data only, then mapped onto the test data. this prevents the model from indirectly seeing test labels through aggregated statistics.

In [5]:
pub_count = df.groupby('publisher').size()
df['publisher_game_count'] = df['publisher'].map(pub_count)

current_year = datetime.now().year
df['game_age'] = current_year - df['release_year']

print(df[['publisher', 'publisher_game_count', 'game_age']].head())

        publisher  publisher_game_count  game_age
0  Rockstar Games                    73      13.0
1  Rockstar Games                    73      12.0
2  Rockstar Games                    73      24.0
3  Rockstar Games                    73      13.0
4      Activision                   523      15.0


## 2. Train / Test Split

The split happends **before** computing the success-rate features to ensure no information from the test set leaks into the training process.

In [16]:
df_clean = df.dropna(subset=['success_combined', 'publisher_encoded', 'genre_encoded', 'console_encoded', 'release_year'])

df_train, df_test = train_test_split(
    df_clean, test_size=0.2, random_state=42, stratify=df_clean['success_combined'])

print(f"Training set: {len(df_train):,} samples")
print(f"Test set: {len(df_test):,} samples")

Training set: 6,899 samples
Test set: 1,725 samples


### Computing Success Rates from Training Data Only

In [23]:
pub_success_rate = df_train.groupby('publisher')['success_combined'].mean()
genre_success_rate = df_train.groupby('genre')['success_combined'].mean()
console_success_rate = df_train.groupby('console')['success_combined'].mean()

global_mean = df_train['success_combined'].mean()

for d in [df_train, df_test]:
    d['publisher_success_rate'] = d['publisher'].map(pub_success_rate).fillna(global_mean)
    d['genre_success_rate'] = d['genre'].map(genre_success_rate).fillna(global_mean)
    d['console_success_rate'] = d['console'].map(console_success_rate).fillna(global_mean)

print("Top 5 publishers by success rate (training set):")
print(pub_success_rate.sort_values(ascending=False).head())

Top 5 publishers by success rate (training set):
publisher
Hello Games                            1.0
Valve                                  1.0
Level 5                                1.0
Sony Computer Entertainment America    1.0
League Of Geeks                        1.0
Name: success_combined, dtype: float64


---
## 3. Final Feature Set

In [32]:
features = [
    'genre_encoded', 'console_encoded', 'publisher_encoded', 'release_year','publisher_game_count', 
    'publisher_success_rate', 'genre_success_rate', 'console_success_rate', 'game_age']

X_train = df_train[features]
y_train = df_train['success_combined']
X_test = df_test[features]
y_test = df_test['success_combined']

print(f"Total features: {len(features)}")
print(f"Class balance: {y_train.mean()*100:.1f}% successful (train), {y_test.mean()*100:.1f}% (test)")

Total features: 9
Class balance: 50.0% successful (train), 50.0% (test)


---
## 4. Feature Scaling

Required for logistic Regression.

In [36]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print("Scaling applied.")

Scaling applied.


---
## 5. Model 1 - Logistic Regression

In [41]:
lr = LogisticRegression(random_state=42, max_iter=2000)
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)
y_prob_lr = lr.predict_proba(X_test_s)[:,1]
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%")
print(f"Logistic Regression AUC-ROC: {roc_auc_score(y_test, y_prob_lr):.4f}")

Logistic Regression Accuracy: 63.77%
Logistic Regression AUC-ROC: 0.6843


In [43]:
print(classification_report(y_test, y_pred_lr, target_names=['Unsuccessful', 'Successful']))

              precision    recall  f1-score   support

Unsuccessful       0.64      0.62      0.63       863
  Successful       0.63      0.66      0.65       862

    accuracy                           0.64      1725
   macro avg       0.64      0.64      0.64      1725
weighted avg       0.64      0.64      0.64      1725



---
## 6 Model 2 - Decision Tree

Depth set to 8 based on tuning experiments for the best balance between underfitting and overfitting.

In [53]:
dt = DecisionTreeClassifier(max_depth=8, random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:,1]

print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt)*100:.2f}%")
print(f"Decision Tree AUC-ROC: {roc_auc_score(y_test, y_prob_dt):.4f}")

Decision Tree Accuracy: 63.94%
Decision Tree AUC-ROC: 0.6733


---
## 7. Model 3 - Random Forest

A Random Forest is an ensemble of many Decision Trees, each trained on a random subset of the data and features. It is usually better than a basic decision tree because it reduces overfitting and captures more diverse patterns. This is the most advanced model in the analysis.

In [56]:
rf = RandomForestClassifier(m_estimators=300, ax_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:,1]

print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf)*100:.2f}%")
print(f"Random Forest AUC-ROC: {roc_auc_score(y_test, y_prob_rf):.4f}")

TypeError: RandomForestClassifier.__init__() got an unexpected keyword argument 'm_estimators'